In [ ]:
import os, sys
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    build_hooks, get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
# nanochat zero-inits c_proj: step-0 attn/mlp.out frames are all-zero; ylim is pinned
# globally so the log-autoscale warning on those frames is pure noise.
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Config this notebook needs (kept out of the generic lib):
BLOCK_REPR = 'block_representations_all'         # all-layer cov runs (the 4 main models)
BLOCK_SAMPLES = 'block_representations_samples'  # samples runs (also carry 160m/410m)
SRC = {                                          # model -> source for spectra/means/eigvals
    'pythia-160m-deduped':  BLOCK_SAMPLES,
    'pythia-410m-deduped':  BLOCK_SAMPLES,
    'pythia-1b-deduped':    BLOCK_REPR,
    'pythia-6.9b-deduped':  BLOCK_REPR,
    'OLMo-2-0425-1B':       BLOCK_REPR,
    'OLMo-2-1124-7B':       BLOCK_REPR,
    'nanochat-d12':         'nanochat_samples',
}
SAMPLES_SRC = {m: 'nanochat_samples' if m == 'nanochat-d12' else BLOCK_SAMPLES for m in SRC}
measured_br = {m: list(range(L)) for m, L in [
    ('pythia-160m-deduped', 12), ('pythia-410m-deduped', 24),
    ('pythia-1b-deduped', 16),   ('pythia-6.9b-deduped', 32),
    ('OLMo-2-0425-1B', 16),      ('OLMo-2-1124-7B', 32),
    ('nanochat-d12', 12)]}
# sequential blocks (OLMo-2, nanochat) expose a distinct mlp.in; parallel Pythia aliases attn.in
has_mlp_in = lambda model: 'olmo' in model.lower() or 'nanochat' in model.lower()
HK = build_hooks()                     # hook-name -> (leaf, metric) table
def bnd(model, prefix, ms):
    return [(SRC[model], HK[f'{prefix}{l}_{m.upper()[:2]}'], f'{m} {l} {prefix}')
            for l in measured_br[model] for m in ms]
attn_in  = lambda model, ms=['Au']: bnd(model, 'AI', ms)
attn_out = lambda model, ms=['Au']: bnd(model, 'AO', ms)
mlp_in   = lambda model, ms=['Au']: bnd(model, 'MI', ms)
mlp_out  = lambda model, ms=['Au']: bnd(model, 'MO', ms)

In [ ]:
# Animated-spectra engine (analysis/spectrum_anim.py): inject this notebook's data
# backend, expose animate_spectra. The notebook drives animations via animate_spectra /
# anim_meancov / anim_blkres (no plot_spectrum needed here).
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
animate_spectra = sa.animate_spectra

## Animated spectra — all boundaries, all models
Profile spectra ($p_j=|\langle\hat\mu, v_j\rangle|^2$) and energy-weighted profiles, animated across checkpoints. Two sections mirroring the source notebook:
- **Mean-covariance** — μ inside the *centered* covariance eigenbasis (`acts_mean_metrics`).
- **Block-vs-residual** — the block-output mean vs the input residual it joins (`mean_metrics_blk_vs_res`, at the sub-block node).

One animation per boundary (profile + weighted, labelled). Run a model's cell to view inline; each animation is ~7 MB. Pass `save_dir='analysis/figures/spectrum_anim'` to `anim_meancov`/`anim_blkres` to render mp4s (fire-and-forget) instead of inline. Smoothing is on by default (`smooth=22, peak=3` — shared `smooth_spectrum`: Savitzky-Golay with an upper-envelope bias that keeps peak heights and edges); pass `smooth=0` to disable.

In [ ]:
from IPython.display import display

_MC_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out), ('MLP in', mlp_in), ('Attn in', attn_in)]
_BR_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out)]

# natural y-scale per metric (matches the source notebook); override via (name, {'ylog': ...})
_METRIC_YLOG = {'mean_norm': True, 'mean_frac': True, 'rayleigh': True, 'mahalanobis': True,
                'pr': True, 'pr_weighted': True, 'rayleigh_normed': False, 'max_overlap': False,
                'top_overlap': False, 'max_overlap_idx': False, 'centroid_idx': False,
                'centroid_idx_weighted': False}

def _u_meancov(model, bnd):                         # legend label = layer number only
    return [(s, (h[0], 'acts_mean_metrics'), lbl.split()[1]) for s, h, lbl in bnd(model, ['Au'])]

def _u_blkres(model, bnd):
    return [(s, (h[0].rsplit('.', 1)[0], 'mean_metrics_blk_vs_res'), lbl.split()[1])
            for s, h, lbl in bnd(model, ['Au'])]

def _strip(m, u, model):                            # m = name or (name, {opts})
    name, mopts = m if isinstance(m, tuple) else (m, {})
    return (name, u, [model], {'kind': 'strip', 'ylog': _METRIC_YLOG.get(name, True), **mopts})

def _anim_section(model, bnds, u_fn, tag, save_dir=None, metrics=(), **kw):
    for name, bnd in bnds:
        u = u_fn(model, bnd)
        panels = [('profile', u, [model], {'title': f'{name} — profile $p_j$'}),
                  *[_strip(m, u, model) for m in metrics],          # thin band between the two
                  ('profile_weighted', u, [model], {'title': f'{name} — weighted'})]
        # smooth: ONLY the mean-to-eigvec overlap profiles are noisy (sorted spectra never are)
        opts = dict(xlog=False, ylog=True, smooth=22, peak=3, model=model,
                    suptitle=f'{tag} — {name} — {model}', **kw)
        safe = f'{tag}_{name}_{model}'.replace(' ', '_')             # save (if any) is a side effect:
        save = f'{save_dir}/{safe}.mp4' if save_dir else None        # always render inline
        display(animate_spectra(panels, save=save, **opts))

def anim_meancov(model, save_dir=None, metrics=(), **kw):
    bnds = [b for b in _MC_BNDS if b[0] != 'MLP in' or has_mlp_in(model)]
    _anim_section(model, bnds, _u_meancov, 'Mean-cov', save_dir, metrics=metrics, **kw)

def anim_blkres(model, save_dir=None, metrics=(), **kw):
    _anim_section(model, _BR_BNDS, _u_blkres, 'Blk-vs-res', save_dir, metrics=metrics, **kw)

### pythia-160m-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-160m-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-160m-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-410m-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-410m-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-410m-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-1b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-1b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-1b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-6.9b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-6.9b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-6.9b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### OLMo-2-0425-1B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-0425-1B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-0425-1B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### OLMo-2-1124-7B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-1124-7B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-1124-7B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### nanochat-d12

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('nanochat-d12', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('nanochat-d12', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

## Animated eigval spectra — MLP outs & stream boundaries

In [ ]:
# Animated centered eigval spectra (nu absolute) per model: one panel of every MLP out,
# one panel of the stream boundaries (attn ins, sequential-arch mlp ins) + before/after final norm.
def anim_eigvals(model, save_dir=None, **kw):
    ac = lambda srcs: [(s, (h[0], 'acts_centered'), ' '.join(lbl.split()[1:])) for s, h, lbl in srcs]
    ins = ac(attn_in(model) + (mlp_in(model) if has_mlp_in(model) else []))
    fins = [(SRC[model], HK.BFN_AC, 'before final norm'), (SRC[model], HK.AFN_AC, 'after final norm')]
    panels = [('eigvals', ac(mlp_out(model)), [model], {'title': r'MLP outs — $\nu$ absolute'}),
              ('eigvals', ins + fins, [model], {'title': r'stream ins + final norms — $\nu$ absolute'})]
    save = f'{save_dir}/Eigvals_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, save=save, xlog=True, ylog=True, model=model,
                            suptitle=f'Eigval spectra — {model}', **kw))

In [ ]:
anim_eigvals('pythia-160m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('pythia-410m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('OLMo-2-1124-7B', save_dir='analysis/figures/animations')

In [ ]:
anim_eigvals('nanochat-d12', save_dir='analysis/figures/animations')

## Animated block-mean cosine heatmap (Exp 1.1)

Pairwise cosine between block-output means (block×block, ordered by depth: attn then mlp per layer), animated across checkpoints. Diagonal hidden. `pin_range=True` (default) holds one symmetric colour range across the whole sweep so colours stay comparable; `pin_range=False` rescales each frame for full contrast. `pair=(rows, cols)` label-substring-selects a sub-block pairing; each model runs the 4 pairings all×all, attn×attn, mlp×mlp, attn×mlp (`PAIRS`).

In [ ]:
# Block×block cosine-of-means heatmap, animated across checkpoints (block_mean_cos per step).
# pin_range=True -> one symmetric colour range over the whole sweep (stable colorbar);
# pin_range=False -> rescale each frame for full per-frame contrast.
# pair=(rows, cols) -> submatrix by label substrings ('' = all); diagonal hidden iff rows == cols.
# save_dir (if given) also writes an mp4 in the background — it does NOT change the inline render.
PAIRS = [('', ''), ('attn', 'attn'), ('mlp', 'mlp'), ('attn', 'mlp')]
def _pair_tag(pair):
    return f"{pair[0] or 'all'}×{pair[1] or 'all'}"

def anim_blockmean_cos(model, save_dir=None, pin_range=True, pair=('', ''), **kw):
    outs = [(SRC[model], HK[f'{p}{l}_AU'], f'{sub} {l}')          # attn then mlp, by depth
            for l in measured_br[model] for p, sub in [('AO', 'attn'), ('MO', 'mlp')]]
    tag = _pair_tag(pair)
    panel = ('acts_mean_vec', outs, [model],
             {'kind': 'heatmap', 'title': f'Block-mean cosine — {tag}',
              'per_frame': not pin_range, 'pair': pair})
    opts = dict(ncols=1, model=model, suptitle=f'Block-mean cosine — {model} — {tag}', **kw)
    save = f"{save_dir}/blockmean_cos_{model}_{tag.replace('×', 'x')}.mp4" if save_dir else None
    display(animate_spectra([panel], save=save, **opts))


In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-160m-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-410m-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-1b-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('pythia-6.9b-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('OLMo-2-0425-1B', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('OLMo-2-1124-7B', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_blockmean_cos('nanochat-d12', save_dir='analysis/figures/animations', pair=pair)

## Animated block↔block coupling (samples run)

Pairwise block-output coupling matrices from the `block_representations_samples` runs (every layer), animated across checkpoints: **CKA** (shared subspace, unsigned), **signed trace** (net reinforce/cancel, energy-weighted), **mean per-token cosine** (democratic counterpart). Covariance-level sequel to the block-mean cosine animation above; matrices are stored per checkpoint, so `kind='matrix'` reads them directly. Same 4-way `pair` pairings as above.

In [ ]:
# Stored (M,M) coupling matrices, animated. cka is unsigned in [0,1] (static range);
# signed trace / mean cos get the symmetric dynamic range (pin_range as above).
# pair=(rows, cols) -> submatrix by label substrings, applied to all three panels.
def anim_block_coupling(model, save_dir=None, pin_range=True, pair=('', ''), src=None, **kw):
    src = src or SAMPLES_SRC[model]
    hook = ('', 'block_block_coupling')
    leaves = get_ys(src, model, hook, 'leaves')[0][0]
    labels = [l.removeprefix('blk').removesuffix('.out') for l in leaves]
    tag = _pair_tag(pair)
    panels = [(y, [(src, hook)], [model],
               {'kind': 'matrix', 'labels': labels, 'title': f'{t} — {tag}',
                'per_frame': not pin_range, 'pair': pair, **o})
              for y, t, o in [('cka', 'CKA', {'dynamic': False, 'vmin': 0, 'vmax': 1}),
                              ('signed_trace', 'signed trace', {}),
                              ('mean_cos', 'mean cos', {})]]
    opts = dict(ncols=3, model=model, suptitle=f'Block↔block coupling — {model} — {tag}', **kw)
    save = f"{save_dir}/block_coupling_{model}_{tag.replace('×', 'x')}.mp4" if save_dir else None
    display(animate_spectra(panels, save=save, **opts))


In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-160m-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-410m-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-1b-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('pythia-6.9b-deduped', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('OLMo-2-0425-1B', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('OLMo-2-1124-7B', save_dir='analysis/figures/animations', pair=pair)

In [ ]:
for pair in PAIRS:
    anim_block_coupling('nanochat-d12', save_dir='analysis/figures/animations', pair=pair)

## Layerwise stream RankMe over training (all models)

The compression valley forming in time: each frame is the depth profile of stream RankMe
(centered, `attn.in` per block + `before_final_norm`). Pythia/nanochat (no write-norm) dig
the canyon at blk4 starting at the spectrum-entropy peak; OLMo-2 never does.


In [ ]:
def anim_layer_rankme(model, save_dir=None, **kw):
    src = SAMPLES_SRC[model]                      # depth profile needs the all-layer samples runs
    layers = [(src, (f'blk{k}.attn.in', 'acts_centered'), f'blk{k}') for k in measured_br[model]]
    layers += [(src, ('before_final_norm', 'acts_centered'), 'bfn')]
    panels = [('rankme', layers, [model], {'kind': 'profile', 'ylog': True,
               'title': 'stream RankMe by depth'})]
    save = f'{save_dir}/layer_rankme_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, ncols=1, save=save, model=model,
                            suptitle=f'Layerwise stream RankMe — {model}', **kw))


In [ ]:
anim_layer_rankme('pythia-160m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('pythia-410m-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('OLMo-2-1124-7B', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme('nanochat-d12', save_dir='analysis/figures/animations')